# Corrective RAG (CRAG)

**Definition:** CRAG improves on traditional RAG by evaluating retrieved documents' relevance before using them, adding a correction step when retrieval quality is poor.

**Workflow:**
Query → Retrieve → Evaluate relevance → Relevant? → Yes: use docs / No: corrective retrieval → Build context → LLM → Answer

**Key components:**

- **Retriever** – fetches candidate documents
- **Evaluator** – scores retrieval relevance
- **Corrective Retrieval** – triggers alternate/additional retrieval if quality is low
- **Context Construction** – assembles final context
- **LLM** – generates answer from corrected context

**CRAG vs Basic RAG**

| Basic RAG                    | CRAG                          |
| ---------------------------- | ----------------------------- |
| Uses retrieved docs directly | Evaluates docs before use     |
| No correction step           | Has correction step           |
| Vulnerable to poor retrieval | Handles poor retrieval better |

**Goal:** Boost RAG reliability by catching bad retrievals and correcting them before generation.


In [61]:
from CRAG import crag

In [62]:
import bs4

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_ollama import OllamaEmbeddings, ChatOllama

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [63]:
# Web bata blog/article load gareko
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

blog_docs = loader.load()

print(f"Loaded documents: {len(blog_docs)}")

Loaded documents: 1


In [64]:
# Thulo document lai sano sano chunks ma divide gareko
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, chunk_overlap=50
)

splits = text_splitter.split_documents(blog_docs)

print(f"Total chunks: {len(splits)}")

Total chunks: 50


In [65]:
# Free/local embedding model use gareko
# Ollama bata nomic-embed-text model use gareko
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Embedding model test gareko
test_embedding = embeddings.embed_query("What is task decomposition?")

print(f"Embedding dimensions: {len(test_embedding)}")

Embedding dimensions: 768


In [66]:
# Document chunks ra embeddings lai Chroma vector database ma store gareko
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

print("Documents indexed successfully!")

Documents indexed successfully!


In [67]:
# User ko question bata top 5 relevant documents retrieve garna retriever banayeko
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Retriever created successfully!")

Retriever created successfully!


In [68]:
# User ko question
question = "what is CRAG?"

# Question bata relevant documents retrieve gareko
retrieved_docs = retriever.invoke(question)

print(f"Retrieved documents: {len(retrieved_docs)}")

# Retrieve bhayeko documents herne
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content[:500])

Retrieved documents: 5

--- Document 1 ---
To avoid overfitting, CoH adds a regularization term to maximize the log-likelihood of the pre-training dataset. To avoid shortcutting and copying (because there are many common words in feedback sequences), they randomly mask 0% - 5% of past tokens during training.
The training dataset in their experiments is a combination of WebGPT comparisons, summarization from human feedback and human preference dataset.

--- Document 2 ---
To avoid overfitting, CoH adds a regularization term to maximize the log-likelihood of the pre-training dataset. To avoid shortcutting and copying (because there are many common words in feedback sequences), they randomly mask 0% - 5% of past tokens during training.
The training dataset in their experiments is a combination of WebGPT comparisons, summarization from human feedback and human preference dataset.

--- Document 3 ---
To avoid overfitting, CoH adds a regularization term to maximize the log-likelihood of the 

In [69]:
# Local LLM ko lagi Ollama ko llama3 model use gareko
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:latest", temperature=0)

In [70]:
# Retrieved document question sanga relevant cha ki chaina
# bhanera check garna prompt banayeko
grader_prompt = ChatPromptTemplate.from_template("""
You are a retrieval relevance grader.

Your task is to determine whether the following
document is relevant to the user's question.

Question:
{question}

Document:
{document}

If the document contains information that can help
answer the question, respond with:

YES

Otherwise respond with:

NO

Respond with only YES or NO.
""")

# Prompt lai LLM sanga connect gareko
retrieval_grader = grader_prompt | llm | StrOutputParser()

In [71]:
# Retrieve bhayeko first document lai test gareko
document = retrieved_docs[0].page_content

# Document relevant cha ki chaina check gareko
grade = retrieval_grader.invoke({"question": question, "document": document})

print("Grade:", grade)

Grade: NO


In [72]:
# Relevant documents matra rakhna empty list banayeko
relevant_docs = []

# Sabai retrieved documents lai ek ek garera check gareko
for i, doc in enumerate(retrieved_docs, start=1):

    # Document question sanga relevant cha ki chaina check gareko
    grade = retrieval_grader.invoke(
        {"question": question, "document": doc.page_content}
    )

    # LLM ko output clean gareko
    grade = grade.strip().upper()

    print(f"Document {i}: {grade}")

    # Relevant document matra list ma rakheko
    if grade == "YES":
        relevant_docs.append(doc)

print(f"\nRelevant documents: {len(relevant_docs)} " f"/ {len(retrieved_docs)}")

Document 1: NO
Document 2: NO
Document 3: NO
Document 4: NO
Document 5: NO

Relevant documents: 0 / 5


In [73]:
# Yedi kunai pani relevant document bhetiyena bhane
# corrective retrieval garne
if len(relevant_docs) == 0:

    print("No relevant documents found.")
    print("Applying corrective retrieval...")

    # Dherai documents retrieve garna naya retriever banayeko
    corrective_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

    # Corrective retrieval gareko
    relevant_docs = corrective_retriever.invoke(question)

else:

    print("Retrieved documents are relevant.")
    print("Using the relevant documents.")

No relevant documents found.
Applying corrective retrieval...


In [74]:
# Relevant documents ko content lai euta context ma combine gareko
context = "\n\n".join(doc.page_content for doc in relevant_docs)

print("Context created.")
print(context[:1000])

Context created.
To avoid overfitting, CoH adds a regularization term to maximize the log-likelihood of the pre-training dataset. To avoid shortcutting and copying (because there are many common words in feedback sequences), they randomly mask 0% - 5% of past tokens during training.
The training dataset in their experiments is a combination of WebGPT comparisons, summarization from human feedback and human preference dataset.

To avoid overfitting, CoH adds a regularization term to maximize the log-likelihood of the pre-training dataset. To avoid shortcutting and copying (because there are many common words in feedback sequences), they randomly mask 0% - 5% of past tokens during training.
The training dataset in their experiments is a combination of WebGPT comparisons, summarization from human feedback and human preference dataset.

To avoid overfitting, CoH adds a regularization term to maximize the log-likelihood of the pre-training dataset. To avoid shortcutting and copying (because

In [75]:
# Retrieved context ko basis ma final answer generate garna prompt banayeko
generation_prompt = ChatPromptTemplate.from_template("""
You are a helpful RAG assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

If the context does not contain enough information
to answer the question, say that you do not have
enough information.

Answer:
""")

# Prompt lai local LLM sanga connect gareko
generation_chain = generation_prompt | llm | StrOutputParser()

In [76]:
# Context ra question LLM lai diyera final answer generate gareko
answer = generation_chain.invoke({"context": context, "question": question})

print(answer)

I don't have enough information to answer this question. The provided context only talks about CoH, WebGPT comparisons, summarization from human feedback and human preference dataset, Long-Term Memory (LTM), and categorization of human memory. There is no mention of CRAG.


#input change garda yestio aaiyo because retrival chai chroma db bata vai rako xa so no info about the question
original prompt was
question = "what is task decomposition for LLM agents?"
